# Stock Prediction and Evaluation (corrected pipeline)

Out-of-sample monthly stock-return prediction with an expanding window: train on return months in years `< t`, predict year `t`.
Predictors measured in month *t* predict the return in month *t+1*.

This notebook is a thin driver around the `stockpred` package (the implementation lives in `stockpred/`; the CLI is `run_linear_backtest.py`).
The original notebook (old timing, annual imputer, row-wise CV) is preserved in `legacy/stock_prediction_and_eval_original.ipynb`; its outputs are in `output_portfolios/`.

Scope of this pass: **OLS, Ridge, RidgeCV, Lasso, LassoCV only**. Neural networks / boosting plug in through `stockpred.runner.make_estimator` and `MODEL_ORDER` later
(note the old NN-saving loop reused one estimator object: `model_dict[m].set_params(**p)` returns the *same* object, so all saved MLPs were identical. The new runner builds a fresh estimator and preprocessor for every fit).

In [ ]:
from pathlib import Path
import pandas as pd
from stockpred import data as D
from stockpred.runner import Backtest, Config
from stockpred.report import evaluate_run

cfg = Config(run_id='linear_v1')   # same defaults the CLI uses


## Data and timing

`Backtest(cfg)` builds the stock-month panel (cached): CRSP returns (percent), OAP signals and the Fama-French factors as predictors.
Signals and predictor factors measured in month *t* get `date = t + 1 month`. The factors used for performance regressions (`ff_eval`) are contemporaneous and are **not** shifted.
The old `ff_factors_lag['date'] - DateOffset(months=1)` pushed month t+1 factor realisations into month t's predictor row; it is replaced by `lag_factors_for_prediction` (+1 month).

In [ ]:
bt = Backtest(cfg)
panel = bt.panel
print(D.assert_timing(panel))                       # raises if signal_date/ff_source_date are not exactly one month before date
ex = panel.loc[panel.signal_date.dt.month == 12, ['permno','signal_date','ff_source_date','date']].head(3)
ex   # December measurement -> January return month

In [ ]:
# Predictor factors (lagged) vs evaluation factors (contemporaneous) are different objects:
print(D.lag_factors_for_prediction(bt.ff_eval).head(3))
print(bt.ff_eval.head(3))

## Preprocessing (fit on training data only)

`stockpred.preprocessing.make_preprocessor` returns a fresh pipeline per fit: missingness filter (training share > 90% missing -> drop) -> same-month cross-sectional mean imputation
(training-sample fallback for fully-missing months; features missing everywhere in training are dropped) -> winsorise (0.25%/99.75%) -> standardise; categoricals: filter -> most-frequent impute -> one-hot.

In [ ]:
pre = bt._preprocess(0, bt.bounds[2000][0])        # fit on return months < 2000 only
print(len(pre.get_feature_names_out()), 'features;', bt._dropped_features(pre))

## Training loop (resumable; separate versioned output directory)

Hyper-parameters for RidgeCV/LassoCV are chosen by chronological validation (last 3 whole years of each training window, expanding), with preprocessing refit in every inner fold.
2020 is an unused buffer year and 2021+ an untouched holdout: those return months are never loaded.
CLI equivalent: `python run_linear_backtest.py --data-dir input_data --run-id linear_v1`.

In [ ]:
bt.run(['OLS'])                                    # smoke/stage 1
bt.run(['Ridge', 'RidgeCV', 'Lasso', 'LassoCV'])   # stage 2 (shares preprocessing per year)

## Portfolios and evaluation

Equal-weighted quintiles with fractional tie allocation (`stockpred.portfolio`), long-short = bin 5 - bin 1 (decimal returns), wealth = `(1+r).cumprod()` ($1 long / $1 short per $1 equity, before costs).
Factor regressions use HAC (Newey-West) errors; RF is subtracted from long-only bins but **not** from long-short spreads.

In [ ]:
perf = evaluate_run(cfg)
out = Path(cfg.out_root) / cfg.run_id / 'results'
perf

In [ ]:
pd.read_csv(out / 'selected_alpha_by_year.csv').query("model in ['RidgeCV','LassoCV']")

In [ ]:
pd.read_csv(out / 'comparison_vs_original.csv')

In [ ]:
print((out / 'report.md').read_text(encoding='utf8')[:6000])